# UCITS Investor Disclosure
**PRIIPs KID — Category 2 Exact Formulas (EU 2017/653 Annex II & IV)**

### VaR in Return Space (Annex II, point 12)

$$\text{VaR}_{\text{RETURN SPACE}} = \sigma\sqrt{N} \cdot \left(-1.96 + 0.474 \cdot \mu_1/\sqrt{N} - 0.0687 \cdot \mu_2/N + 0.146 \cdot \mu_1^2/N\right) - 0.5\sigma^2 N$$

Where:
- $\sigma$ = volatility per period
- $N$ = number of trading periods in RHP
- $\mu_1$ = skew
- $\mu_2$ = excess kurtosis

### VEV (Annex II, point 13)

$$\text{VEV} = \left\{\sqrt{3.842 - 2 \cdot \text{VaR}_{\text{RETURN SPACE}}} - 1.96\right\} / \sqrt{T}$$

Where $T$ is the RHP in years.

Note: 3.842 = 1.96², the squared z-value at 97.5% confidence.

---

### Performance Scenarios (Annex IV)

1. Unfavourable (10th percentile equivalent)

$$\text{Exp}\left[M1 \cdot N + \sigma\sqrt{N} \cdot \left(-1.28 + 0.107 \cdot \mu_1/\sqrt{N} + 0.0724 \cdot \mu_2/N - 0.0611 \cdot \mu_1^2/N\right) - 0.5\sigma^2 N\right]$$
<br>

2. Moderate (median)

$$\text{Exp}\left[M1 \cdot N - \sigma\mu_1/6 - 0.5\sigma^2 N\right]$$

<br>

3. Favourable (90th percentile equivalent)

$$\text{Exp}\left[M1 \cdot N + \sigma\sqrt{N} \cdot \left(1.28 + 0.107 \cdot \mu_1/\sqrt{N} - 0.0724 \cdot \mu_2/N + 0.0611 \cdot \mu_1^2/N\right) - 0.5\sigma^2 N\right]$$

<br>

4. Stress Scenario

- **Step 1**: compute rolling sub-period volatility ${}^w\sigma_S$ over window $w$:

    $${}^w_{t_i}\sigma_S = \sqrt{\frac{\sum_{t_i}^{t_i+w}\left(r_{t_i} - {}^{t_i+w}_{t_i}M_1\right)^2}{M_w}}$$

- **Step 2**: infer stressed volatility ${}^w\sigma_S$ as:
    - 99th percentile of sub-period vols for 1Y RHP
    - 90th percentile for other RHP

- **Step 3**: stress scenario value:

    $$\text{Scenario}_{\text{Stress}} = exp{\left[{}^w\sigma_S \cdot \sqrt{N} \cdot \left(z_\alpha + \frac{z_\alpha^2-1}{6} \cdot \frac{\mu_1}{\sqrt{N}} + \frac{z_\alpha^3 - 3z_\alpha}{24} \cdot \frac{\mu_2}{N} - \frac{2z_\alpha^3 - 5z_\alpha}{36} \cdot \frac{\mu_1^2}{N}\right) - 0.5 \cdot {}^w\sigma_S^2 N\right]}$$

    Where $z_\alpha$ corresponds to:
    - 1% percentile for 1Y RHP
    - 5% percentile for other RHP

---

### Regulatory Constants Summary

<small>

| Constant | Value | Used in |
|---|---|---|
| VaR confidence | 97.5% | VaR, VEV |
| z at 97.5% | 1.96 | VaR formula |
| 1.96² | 3.842 | VEV formula |
| Unfavourable z | 1.28 (negative) | Unfavourable scenario |
| Favourable z | 1.28 (positive) | Favourable scenario |
| Stress percentile (1Y) | 99th sub-period vol | Stress scenario |
| Stress percentile (other) | 90th sub-period vol | Stress scenario |
| Stress z (1Y) | 1% percentile | Stress scenario |
| Stress z (other) | 5% percentile | Stress scenario |
| Cornish-Fisher coefficients | 0.474, 0.0687, 0.146 | VaR formula |
| Scenario CF coefficients | 0.107, 0.0724, 0.0611 | Scenario formulas |
```

The performance of the PRIIP shall be calculated net of all applicable costs in accordance with Annex VI for the scenario and holding period being presented.
32.

Performance shall be presented in monetary units. The amounts used shall be consistent with the amounts referred to in point 90 of Annex VI.
33.

Performance shall also be presented in percentage terms, as the average annual return of the investment. That figure shall be calculated considering net performance as numerator and the initial investment amount or the price paid as denominator.

In [ ]:
from src.config import VALUATION_DATE, QUARTER
from src.risk.reg_constants import CONFIDENCE, HORIZON, NOTICE, GROSS_LIMIT

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src.ui.plot_style import (
    C, FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
    apply_ax_style, section_title, fig_header, fig_footer,
    callout_box, threshold_vline, threshold_hline, breach_fill,
    pct_color, rag_color, util_color, liq_color, table_cell, sup_title,
)
from src.ui.nb_utils import save_fig, save_table, styled_table, save_table_png

import warnings
warnings.filterwarnings('ignore')

from src.setup_db import run as setup_db
setup_db()

from src.data.database import get_engine, query_positions, query_nav_history
from src.data.enrichment import enrich_positions, get_risk_ready_df
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
from src.risk.risk_utils import (
    var_historical, var_parametric, var_scale,
    es_historical, es_parametric, es_scale,
    kupiec_test, christoffersen_test,
    exception_report, full_backtest_report,
    stress_equity, stress_rates, stress_credit,
    stress_fx, stress_combined, stress_historical,
    days_to_liquidate, liquidity_buckets, redemption_stress, investor_concentration,
)
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml
import src.print_utils as prt
import src.risk.esg_utils as esg_u
from src.risk.esg_utils import ESG_FIELDS, ESG_THRESHOLD_LOW, ESG_THRESHOLD_HIGH


# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
FUND_ID    = 'UCITS_Balanced'
VALUATION_DATE      = '2026-05-13'
ENGINE     = get_engine()
BBG        = Bloomberg()

# UCITS regulatory limits
VAR_LIMIT_ABS = 0.20   # 20% of NAV (absolute VaR limit)
VAR_LIMIT_REL = 2.0    # 2x reference portfolio (relative VaR limit)
CONFIDENCE    = 0.99   # 99% confidence level
HORIZON       = 20     # 20 trading days (UCITS standard)

print(f'Fund      : {FUND_ID}')
print(f'Date      : {VALUATION_DATE}')
print(f'VaR limit : {VAR_LIMIT_ABS*100:.0f}% NAV (absolute)')
print(f'VaR limit : {VAR_LIMIT_REL:.0f}x reference (relative)')
print(f'Confidence: {CONFIDENCE*100:.0f}%')
print(f'Horizon   : {HORIZON} days')

## 1. Load and Validate Positions

Load today's UCITS Balanced Fund positions from the daily fund
administrator export and validate UCITS-specific requirements.
Unlike AIFM funds, UCITS imposes strict eligibility and
concentration rules that must be checked at the position level
before any risk calculation begins.

In [ ]:
# query from SQLdb
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)

# enrich w/ info from Bloomberg
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# info in risk_df covers many risk inputs
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV)

In [ ]:
#

# ----------------------------------------------------------------
# UCITS specific validations
# ----------------------------------------------------------------
print(f'\n--- UCITS compliance checks ---')

# 1. all positions long only (UCITS cannot short)
has_shorts = (positions['market_value_eur'] < 0).any()
print(f'  Long only        : {"FAIL - short positions detected" if has_shorts else "OK"}')

# 2. no single position > 10% NAV
# ETFs are exempt: they are diversified instruments
# compute weight_abs first
positions['weight_abs'] = positions['market_value_eur'].abs() / NAV * 100

# then filter non-ETF
non_etf     = positions[positions['sub_asset_class'] != 'ETF'].copy()
breaches_10 = non_etf[non_etf['weight_abs'] > 10]

if len(breaches_10) > 0:
    print(f'  10% limit        : FLAG - {len(breaches_10)} non-ETF positions exceed limit')
    for _, row in breaches_10.iterrows():
        print(f'    {row["instrument_name"]:<30}: {row["weight_abs"]:.1f}%')
else:
    print(f'  10% limit        : OK (ETFs exempt as diversified instruments)')

# 3. all instruments UCITS eligible (listed, liquid)
illiquid = positions[positions['adv_eur'] == 0]
illiquid = illiquid[illiquid['asset_class'] != 'Cash']
if len(illiquid) > 0:
    print(f'  Liquidity        : FLAG - {len(illiquid)} illiquid instruments')
    for _, row in illiquid.iterrows():
        print(f'    {row["instrument_name"]}')
else:
    print(f'  Liquidity        : OK - all instruments liquid')
# 4. weights sum to 100%
weight_sum = positions['weight_pct'].sum()
print(f'  Weights sum      : {weight_sum:.2f}% '
      f'({"OK" if abs(weight_sum - 100) < 1 else "FLAG"})')

In [ ]:
# Asset class breakdown
breakdown = risk_df.groupby('asset_class').agg(
    market_value_eur=('market_value_eur', 'sum'),
    n_positions=('isin', 'count'),
).sort_values('market_value_eur', ascending=False)

breakdown['weight_pct'] = breakdown['market_value_eur'] / NAV * 100
# prt.print_asset_class_weights_n_positions(breakdown, NAV)
phtml.display_asset_class_weights_n_positions(breakdown, NAV)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 2))
colors = [ACCENT2 if v < 0 else ACCENT for v in breakdown['weight_pct']]

bars = ax.barh(breakdown.index, breakdown['weight_pct'],
               color=colors, height=0.45, alpha=0.85)

ax.set_xlabel('Weight (% NAV)', fontsize=9)
ax.set_xlim(0, breakdown['weight_pct'].max() * 1.1)
section_title(ax, 'Asset Class Breakdown')

for bar, val in zip(bars, breakdown['weight_pct']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=7)
    
ax.grid(False)
plt.tight_layout()
save_fig(fig, FUND_ID, "01. Asset Breakdown HF", dpi=150)
plt.show()

## 2. Absolute VaR

The UCITS absolute VaR limit requires that the 99% confidence,
20-day VaR does not exceed **20% of NAV** at any point. This is
monitored daily and reported to the CSSF on request.

$$VaR_{20d}^{99\%} = VaR_{1d}^{99\%} \cdot \sqrt{20}$$

Two methods are computed and compared:
- **Historical simulation**: no distribution assumption, uses actual
  fund P&L history (pulls from empirical quantile)
- **Parametric (Student-t)**: uses the sample mean $\mu$ and
  standard deviation $\sigma$ estimated from the 250-day P&L
  history as inputs to the Student-t quantile formula

> **Regulatory note**: CESR/10-788 permits three VaR methodologies for UCITS:
> variance-covariance (parametric), historical simulation, and Monte Carlo simulation.
> The choice is the responsibility of the ManCo and must be documented in the RMP.
> This notebook implements historical simulation and parametric VaR.
> Monte Carlo simulation is not implemented here but would be required for portfolios
> with significant non-linear exposure (e.g. options, structured products).

**Data window**: both methods use the last 250 trading days (~1 year)
of daily fund P&L as the estimation window. This is the regulatory
minimum under UCITS. A longer window captures more market regimes
but is slower to react to current conditions; a shorter window is
more reactive but less stable.

For historical simulation, 250 observations give 2-3 data points
in the 1% tail, which is the minimum for a reliable quantile estimate.
For parametric VaR, the same window is used to estimate $\mu$ and
$\sigma$ which feed into the Student-t formula.

**New funds**: when the fund has less than 250 days of history,
the standard practice is to use a **proxy portfolio** with the
same asset class composition and weights applied to a benchmark
index with sufficient history. The CSSF accepts this approach
for funds in their first year of operation, provided the proxy
methodology is documented in the risk management policy.

In [ ]:
# ----------------------------------------------------------------
# Absolute VaR computation
# ----------------------------------------------------------------

# get P&L history from NAV
nav_history = query_nav_history(ENGINE, FUND_ID)
returns         = nav_history['pnl_pct'].dropna().values

print(f'P&L history: {len(returns)} daily observations')
print(f'Mean daily return : {returns.mean()*100:.4f}%')
print(f'Daily volatility  : {returns.std()*100:.4f}%')
print(f'Ann. volatility   : {returns.std()*np.sqrt(252)*100:.2f}%')

# --- Historical simulation VaR ---
var_hist_1d  = var_historical(returns, confidence=CONFIDENCE)
var_hist_20d = var_scale(var_hist_1d, horizon=HORIZON)

# --- Parametric VaR (Student-t) ---
mu           = returns.mean()
sigma        = returns.std()
var_para_1d  = var_parametric(mu=mu, sigma=sigma,
                               confidence=CONFIDENCE, dist='t', df=5)
var_para_20d = var_scale(var_para_1d, horizon=HORIZON)


print(f'\n--- VaR results ---')
print(f'{"Method":<20} {"1-day":>10} {"20-day":>10} '
      f'{"Limit":>10} {"Utilization":>12} {"Status":>8}')
print('-' * 75)

for method, var_1d, var_20d in [
    ('Historical',  var_hist_1d,  var_hist_20d),
    ('Parametric',  var_para_1d,  var_para_20d),
]:
    utilization = var_20d / VAR_LIMIT_ABS * 100
    status      = ('🔴 BREACH' if var_20d > VAR_LIMIT_ABS else
                   # 80% threshold for warning zone = var 16%+
                   '🟡 WARNING' if var_20d > VAR_LIMIT_ABS * 0.80 else
                   '🟢 OK')
    print(f'{method:<20} {var_1d*100:>9.3f}% {var_20d*100:>9.3f}% '
          f'{VAR_LIMIT_ABS*100:>9.0f}% {utilization:>11.1f}% {status:>8}')



# ----------------------------------------------------------------
# Rolling VaR chart
# ----------------------------------------------------------------
window   = 60
rolling_var = pd.Series([
    var_historical(returns[max(0, i-window):i], confidence=CONFIDENCE)
    for i in range(window, len(returns)+1)
])
dates_plot = nav_history['date'].iloc[window:].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 5))
sup_title(fig, 'UCITS Absolute VaR — 60-day Rolling Window', fontsize=18)


ax.plot(dates_plot, rolling_var * np.sqrt(HORIZON) * 100,
        color=ACCENT, lw=1.5, label='20-day VaR (%)')
ax.axhline(VAR_LIMIT_ABS * 100, color='red', lw=1.5,
           ls='--', label=f'UCITS limit ({VAR_LIMIT_ABS*100:.0f}%)')
ax.axhline(VAR_LIMIT_ABS * 80, color=ACCENT2, lw=1,
           ls='--', label='Warning (80% of limit)')
ax.fill_between(dates_plot,
                rolling_var * np.sqrt(HORIZON) * 100,
                VAR_LIMIT_ABS * 100,
                where=rolling_var * np.sqrt(HORIZON) > VAR_LIMIT_ABS,
                color='red', alpha=0.3, label='Breach zone')
ax.set_ylabel('20-day VaR (% of NAV)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, FUND_ID, '01. VaR 60d - UCITS')
plt.show()

The rolling VaR chart below is for internal monitoring only.
The single regulatory VaR figure computed from the full 250-day
window is what is reported to the CSSF. The rolling chart shows
how the risk profile has evolved over time, making gradual
increases in portfolio risk visible before they become a
regulatory concern.

## 3. Expected Shortfall (ES)
**Expected Shortfall (ES)** measures the average loss in the worst
scenarios beyond the VaR threshold. It is teh conditional expected loss when VAR is breached.

$$ES_{\alpha} = -\mathbb{E}[R \mid R < -VaR_{\alpha}]$$

For UCITS reporting, historical ES is the standard. Parametric ES
under the Student-t distribution:

$$ES_{\alpha} = \sigma \cdot \frac{f_t(t_{\alpha}) \cdot (\nu + t_{\alpha}^2)}{(\nu - 1)(1 - \alpha)}$$

is shown below for completeness. Parametric ES is not required by UCITS or AIFMD regulation and is more commonly encountered in the banking context under Basel IV, where Expected Shortfall at 97.5% replaced VaR as the primary capital measure under FRTB.

In [ ]:
# --- ES ---
es_hist_1d   = es_historical(returns, confidence=CONFIDENCE)
es_hist_20d  = es_scale(es_hist_1d, horizon=HORIZON)

# ----------------------------------------------------------------
# Expected Shortfall: parametric (Student-t) for completeness
# ----------------------------------------------------------------

es_para_1d  = es_parametric(sigma=sigma, mu=mu,
                             confidence=CONFIDENCE, dist='t', df=5)
es_para_20d = es_scale(es_para_1d, horizon=HORIZON)

print('--- Expected Shortfall comparison ---')
print(f'{"Method":<20} {"ES 1-day":>10} {"ES 20-day":>10} {"ES/VaR":>8}')
print('-' * 52)
print(f'{"Historical":<20} {es_hist_1d*100:>9.3f}% '
      f'{es_hist_20d*100:>9.3f}% '
      f'{es_hist_1d/var_hist_1d:>7.2f}x')
print(f'{"Parametric (t)":<20} {es_para_1d*100:>9.3f}% '
      f'{es_para_20d*100:>9.3f}% '
      f'{es_para_1d/var_para_1d:>7.2f}x')

print(f'\nNote: ES > VaR always holds by construction.')
print(f'ES/VaR ratio reflects tail heaviness: '
      f'higher ratio = fatter tails.')

## 4. Relative VaR

The UCITS relative VaR limit requires that the fund VaR does not
exceed **2x the VaR of the reference portfolio**. This approach
is used when the fund has a clear benchmark and is more appropriate
than the absolute limit for funds whose risk profile is defined
relative to a market index.

$$\text{Relative VaR} = \frac{VaR_{fund}}{VaR_{reference}} \leq 2$$

The reference portfolio for UCITS Balanced is a standard 60/40
allocation: 60% MSCI World (proxied by SPY) and 40% EUR government
bonds (proxied by IEAG, iShares Core EUR Govt Bond ETF). This is
documented in the fund's risk management policy and approved by
the CSSF.

In [ ]:
# ----------------------------------------------------------------
# Relative VaR
# ----------------------------------------------------------------

# reference portfolio: 60% SPY, 40% TLT
# reference portfolio: 60% SPY (MSCI World proxy), 40% IEAG (EUR Govt Bond proxy)
w_eq  = 0.60
w_bd  = 0.40

spy_hist  = BBG.bdh('SPY US Equity', 'PX_LAST', '2025-05-13', VALUATION_DATE)
ieag_hist = BBG.bdh('IEAG LN Equity', 'PX_LAST', '2025-05-13', VALUATION_DATE)

spy_ret  = spy_hist['PX_LAST'].pct_change().dropna()
ieag_ret = ieag_hist['PX_LAST'].pct_change().dropna()

ref_ret = (w_eq * spy_ret + w_bd * ieag_ret).dropna().values

print(f'Reference portfolio: {w_eq*100:.0f}% SPY + {w_bd*100:.0f}% TLT')
print(f'Reference history  : {len(ref_ret)} days')
print(f'Reference daily vol: {ref_ret.std()*100:.4f}%')
print(f'Fund daily vol     : {returns.std()*100:.4f}%')

# VaR for reference portfolio
var_ref_1d  = var_historical(ref_ret, confidence=CONFIDENCE)
var_ref_20d = var_scale(var_ref_1d, horizon=HORIZON)

# relative VaR ratio
relative_var = var_hist_20d / var_ref_20d
status       = ('🔴 BREACH' if relative_var > VAR_LIMIT_REL else
                '🟡 WARNING' if relative_var > VAR_LIMIT_REL * 0.80 else
                '🟢 OK')

print(f'\n--- Relative VaR ---')
print(f'Fund VaR 20d      : {var_hist_20d*100:.3f}%')
print(f'Reference VaR 20d : {var_ref_20d*100:.3f}%')
print(f'Relative VaR ratio: {relative_var:.2f}x '
      f'(limit: {VAR_LIMIT_REL:.0f}x)  {status}')
print(f'Utilization       : {relative_var/VAR_LIMIT_REL*100:.1f}% of limit')

# ----------------------------------------------------------------
# Chart: fund vs reference portfolio rolling VaR
# ----------------------------------------------------------------
window = 60
min_len = min(len(returns), len(ref_ret))
returns_aligned = returns[-min_len:]
ref_aligned = ref_ret[-min_len:]

rolling_var_fund = pd.Series([
    var_historical(returns_aligned[max(0,i-window):i], CONFIDENCE)
    for i in range(window, min_len+1)
])
rolling_var_ref = pd.Series([
    var_historical(ref_aligned[max(0,i-window):i], CONFIDENCE)
    for i in range(window, min_len+1)
])
rolling_ratio = rolling_var_fund / rolling_var_ref

# dates_rel     = nav_history['date'].iloc[window:].reset_index(drop=True)
dates_rel = nav_history['date'].iloc[-len(rolling_var_fund):].reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
sup_title(fig, 'Relative VaR: Fund vs Reference Portfolio', fontsize=18)

ax1.plot(dates_rel, rolling_var_fund * np.sqrt(HORIZON) * 100,
         color=ACCENT, lw=1.5, label='Fund VaR 20d')
ax1.plot(dates_rel, rolling_var_ref * np.sqrt(HORIZON) * 100,
         color=ACCENT3, lw=1.5, label='Reference VaR 20d')
ax1.set_ylabel('20-day VaR (%)')
ax1.legend(fontsize=8)

ax2.plot(dates_rel, rolling_ratio,
         color=C['cyan'], lw=1.5, label='Relative VaR ratio')
ax2.axhline(VAR_LIMIT_REL, color=C['red'], lw=1.5,
            ls='--', label=f'Limit ({VAR_LIMIT_REL:.0f}x)')
ax2.axhline(VAR_LIMIT_REL * 0.80, color=C['amber'], lw=1,
            ls='--', label='Warning (80% of limit)')
ax2.set_ylabel('Relative VaR ratio')
ax2.legend(fontsize=8)

plt.tight_layout()
save_fig(fig, FUND_ID, '02. Relative VaR vs. ref portfolio')
plt.show()


## 5. SRRI Computation
The SRRI (Synthetic Risk and Reward Indicator) is computed from 5 years of weekly NAV returns,
annualised, and mapped to a 1–7 category per CESR/10-673.

CESR/10-673 is the guidelines document published by the Committee of European Securities Regulators (the predecessor to ESMA) that defines exactly how to compute the SRRI. It specifies 
* the 5-year weekly return window
* the annualisation formula
* seven volatility buckets that map to categories 1 through 7. 

It is the regulatory basis for the risk indicator shown on every UCITS KIID.

| SRRI | Volatility range |
|------|-----------------|
| 1    | < 0.5%          |
| 2    | 0.5% – 2%       |
| 3    | 2% – 5%         |
| 4    | 5% – 10%        |
| 5    | 10% – 15%       |
| 6    | 15% – 25%       |
| 7    | >= 25%          |

In [ ]:
# MRS-41: SRRI computation (CESR/10-673)
nav_history_full = query_nav_history(ENGINE, FUND_ID)
nav_history_full['date'] = pd.to_datetime(nav_history_full['date'])
nav_history_full = nav_history_full.set_index('date')

# CESR/10-673 mandates exactly 260 weekly observations (5 years)
# last('260W') was removed in pandas 2.0; use explicit date filter instead
cutoff = nav_history_full.index.max() - pd.DateOffset(weeks=260)
nav_history_5y = nav_history_full[nav_history_full.index >= cutoff]

weekly_nav = nav_history_5y['nav_eur'].resample('W').last()
weekly_ret = weekly_nav.pct_change().dropna()
sigma_weekly = weekly_ret.std()
sigma_ann = sigma_weekly * np.sqrt(52)

def compute_srri(sigma_ann: float) -> int:
    """Map annualised volatility to SRRI category per CESR/10-673 Table 1.

    Boundaries: 1:<0.5%, 2:0.5-2%, 3:2-5%, 4:5-10%, 5:10-15%, 6:15-25%, 7:>=25%
    """
    boundaries = [0, 0.005, 0.02, 0.05, 0.10, 0.15, 0.25]
    for i, bound in enumerate(boundaries):
        if sigma_ann <= bound:
            return i
    return 7

srri = compute_srri(sigma_ann)

print(f"Weekly return observations : {len(weekly_ret)}")
print(f"Annualised volatility      : {sigma_ann * 100:.2f}%")
print(f"SRRI category              : {srri}")

## 6. UCITS Stress Testing

UCITS regulations (2009/65/EC and ESMA guidelines) require stress testing but do not
prescribe specific scenario parameters. Shocks below are defined in the fund's Risk
Management Policy (RMP) and approved by the board.

Scenarios covered:
- **Equity crash**: equity down 30%
- **Rate shock**: parallel shift up 200bps
- **Credit widening**: credit spreads widen 150bps
- **FX stress**: USD/GBP depreciate 15% vs EUR
- **Combined**: simultaneous equity, rate, credit and FX shock
- **Historical**: 2008 financial crisis, 2020 COVID crash, 2022 rate shock

In [ ]:
# MRS-42: stress testing setup
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)
NAV     = risk_df['market_value_eur'].sum()


In [ ]:
# Equity crash: -30%
eq = stress_equity(risk_df, delta_equity=-0.30)
eq_pct = eq['stressed_pnl_eur'] / NAV * 100

# Rate shock: +200bps
rt = stress_rates(risk_df, delta_y=0.02)
rt_pct = rt['stressed_pnl_eur'] / NAV * 100

# Credit widening: +150bps
cr = stress_credit(risk_df, delta_spread=0.015)
cr_pct = cr['stressed_pnl_eur'] / NAV * 100

# FX stress: USD and GBP depreciate 15% vs EUR
fx = stress_fx(risk_df, fx_shocks={'USD': -0.15, 'GBP': -0.15})
fx_pct = fx['stressed_pnl_eur'] / NAV * 100

summary = pd.DataFrame([
    {'Scenario': 'Equity Crash -30%',      'P&L (EUR)': eq['stressed_pnl_eur'], '% NAV': eq_pct},
    {'Scenario': 'Rate Shock +200bps',     'P&L (EUR)': rt['stressed_pnl_eur'], '% NAV': rt_pct},
    {'Scenario': 'Credit Widening +150bps','P&L (EUR)': cr['stressed_pnl_eur'], '% NAV': cr_pct},
    {'Scenario': 'FX Stress USD/GBP -15%', 'P&L (EUR)': fx['stressed_pnl_eur'], '% NAV': fx_pct},
])

summary['P&L (EUR)'] = summary['P&L (EUR)'].map('{:,.0f}'.format)
summary['% NAV']     = summary['% NAV'].map('{:.2f}%'.format)
summary.set_index('Scenario', inplace=True)
summary

In [ ]:
# Combined scenario
cb = stress_combined(risk_df)
cb_pct = cb['stressed_pnl_eur'] / NAV * 100

components = {
    'Equity': cb['equity_pnl'] / NAV * 100,
    'Rates':  cb['rates_pnl']  / NAV * 100,
    'Credit': cb['credit_pnl'] / NAV * 100,
    'FX':     cb['fx_pnl']     / NAV * 100,
}

# Historical scenarios
scenarios = ['2008', '2020', '2022']
labels    = ['2008 Financial Crisis', '2020 COVID Crash', '2022 Rate Shock']
results   = [stress_historical(risk_df, s) for s in scenarios]
pnls_pct  = [r['stressed_pnl_eur'] / NAV * 100 for r in results]

# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5),
                                gridspec_kw={'width_ratios': [4, 3]})

# ── left: combined scenario ──────────────────────────────────────────────────
bars1 = ax1.bar(components.keys(), components.values(),
                color=[ACCENT2 if v < 0 else ACCENT3 for v in components.values()],
                width=0.45, alpha=0.85)
ax1.axhline(0, color=C['dim'], lw=0.8)
ax1.set_ylabel('P&L impact (% NAV)', fontsize=11)
ax1.tick_params(axis='both', labelsize=11)

# right
section_title(ax1, f'Combined Stress Total: {cb_pct:.2f}% NAV', fontsize=12)
for bar, val in zip(bars1, components.values()):
    y = val - 0.3 if abs(val) < 2 else val / 2
    va = 'top' if abs(val) < 2 else 'center'
    ax1.text(bar.get_x() + bar.get_width() / 2, y,
             f'{val:.2f}%', ha='center', va=va,
             fontsize=11, color='white', fontweight='bold')

# ── right: historical scenarios ──────────────────────────────────────────────
bars2 = ax2.bar(labels, pnls_pct,
                color=[ACCENT2 if v < 0 else ACCENT3 for v in pnls_pct],
                width=0.45, alpha=0.85)
ax2.axhline(0, color=C['dim'], lw=0.8)
ax2.set_ylabel('P&L impact (% NAV)', fontsize=11)
ax2.tick_params(axis='both', labelsize=11)
section_title(ax2, 'Historical Stress Scenarios', fontsize=12)
for bar, val in zip(bars2, pnls_pct):
    ax2.text(bar.get_x() + bar.get_width() / 2, val / 2,
             f'{val:.2f}%', ha='center', va='center',
             fontsize=11, color='white', fontweight='bold')

sup_title(fig, f'Stress Tests — {FUND_ID}  |  {VALUATION_DATE}', fontsize=18)
plt.tight_layout()
save_fig(fig, FUND_ID, '03. Stress tests: shock, combined, historical')
plt.show()

print(f"Combined stress P&L: EUR {cb['stressed_pnl_eur']:,.0f}  ({cb_pct:.2f}% NAV)")
for label, r, pct in zip(labels, results, pnls_pct):
    print(f"{label:30s}: EUR {r['stressed_pnl_eur']:>15,.0f}  ({pct:.2f}% NAV)")

> **Methodology note**: stress P&L is computed using first-order sensitivities
> (delta for equities, modified duration for rates and credit, direct revaluation for FX).
> Convexity and cross-gamma effects are not captured. In production, a ManCo would
> source these figures from a risk system (Bloomberg PORT, Aladdin, Axioma) which
> performs full revaluation. Results here should be read as directional estimates.

## 7. VaR Backtest (Kupiec + Christoffersen + ESMA)

VaR backtesting compares predicted VaR against realised daily P&L.
Two statistical tests are applied:

- **Kupiec POF**: tests whether the breach rate equals the expected rate (1% for 99% VaR)
- **Christoffersen**: tests whether breaches are independently distributed over time

> **Note**: Kupiec and Christoffersen tests are industry best practice and not explicitly
> required by ESMA or CSSF. The regulatory requirement is the 250-day breach count
> and zone classification only. Both tests should be documented in the RMP. For both models 
> p > 0.05: fail to reject the null, model is statistically acceptable

ESMA (CESR/10-788) requires backtesting 1-day 99% VaR against realised daily P&L
over a 250 trading day rolling window. Both historical and parametric VaR are acceptable;
historical simulation is preferred in practice as it makes no distributional assumptions
and is easier to justify to the CSSF.

Zone classification (Basel traffic light, adopted by ESMA):
- **Green** (0-4 breaches): model acceptable
- **Amber** (5-9 breaches): explanation required, possible model review
- **Red** (10+ breaches): model must be revised, CSSF notification required

>Note: POF=Proportion of Failures. It tests whether the observed breach rate (number of days P&L exceeded VaR divided by total observations) is statistically consistent with the expected rate (1% for 99% VaR). It is a likelihood ratio test.

#### 7.1. Internal usage: full period

In [ ]:
# MRS-43: VaR backtest
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])

returns    = nav_history['pnl_pct'].dropna().values
dates  = nav_history['date'].iloc[1:].reset_index(drop=True)

# compute rolling 1-day VaR series (250-day window)
window = 250
var_hist  = pd.Series([
    var_historical(returns[max(0, i-window):i], confidence=0.99)
    for i in range(window, len(returns))
])
var_param = pd.Series([
    var_parametric(mu=0, sigma=returns[max(0, i-window):i].std(),
                   confidence=0.99, dist='t')
    for i in range(window, len(returns))
])

returns_aligned   = returns[window:]
dates_aligned = dates.iloc[window:].reset_index(drop=True)

print(f"Backtest observations : {len(returns_aligned)}")

In [ ]:
report = full_backtest_report(
    returns_series=pd.Series(returns_aligned),
    var_dict={'Historical': var_hist, 'Parametric': var_param},
    dates=dates_aligned
)

report[['n_obs', 'n_breaches', 'breach_rate', 'expected',
        'kupiec_p', 'christoffersen_p', 'result']]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dates_aligned, returns_aligned * 100,
        color=C['muted'], lw=0.8, label='Daily P&L', alpha=0.7)
ax.plot(dates_aligned, -var_hist * 100,
        color=ACCENT, lw=1.2, label='99% VaR (historical)')
ax.plot(dates_aligned, -var_param * 100,
        color=ACCENT3, lw=1.2, label='99% VaR (parametric)', linestyle='--')

breaches = returns_aligned < -var_hist.values
ax.scatter(dates_aligned[breaches], returns_aligned[breaches] * 100,
           color=ACCENT2, s=10, zorder=5, label=f'Breaches ({breaches.sum()})')

ax.set_ylabel('Daily P&L / VaR (%)', fontsize=9)
section_title(ax, f'VaR Backtest — {FUND_ID}', fontsize=12)
ax.legend(fontsize=10, loc="lower right")
plt.tight_layout()
save_fig(fig, FUND_ID, "04. VAR backtest UCITS - full history")
plt.show()

#### 7.2. ESMA report 250 days

In [ ]:
# ESMA exception report: last 250 trading days (regulatory window)
esma_250    = exception_report(pd.Series(returns_aligned[-250:]), var_hist.iloc[-250:], confidence=0.99)
n_250       = len(esma_250)
breach_250  = n_250 / 250

if n_250 <= 4:
    zone_250 = 'Green'
elif n_250 <= 9:
    zone_250 = 'Amber'
else:
    zone_250 = 'Red'

# Display ESMA backtest summary
esma_summary = pd.DataFrame([
    {'Metric': 'Observation window', 'Value': '250 trading days (ESMA regulatory)'},
    {'Metric': 'Breaches', 'Value': f'{n_250}'},
    {'Metric': 'Breach rate', 'Value': f'{breach_250*100:.2f}%'},
    {'Metric': 'Expected rate', 'Value': '1.0%'},
    {'Metric': 'ESMA zone', 'Value': zone_250},
]).set_index('Metric')
# phtml.display_dark_table(esma_summary, caption='ESMA VaR Backtest Report (250 days)')

phtml.display_dark_table(esma_250, caption='Exception Details')

In [ ]:
dates_250 = dates_aligned.iloc[-250:].reset_index(drop=True)
returns_250   = returns_aligned[-250:]
var_250   = var_hist.iloc[-250:].reset_index(drop=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(dates_250, returns_250 * 100,
        color=C['muted'], lw=1.2, label='Daily P&L', alpha=0.7)
ax.plot(dates_250, -var_250 * 100,
        color=ACCENT, lw=1.2, label='99% VaR (historical)')

breaches_250 = returns_250 < -var_250.values
ax.scatter(dates_250[breaches_250], returns_250[breaches_250] * 100,
           color=ACCENT2, s=30, zorder=5, label=f'Breaches ({n_250})')

zone_color = {'Green': C['green'], 'Amber': C['amber'], 'Red': C['red']}

section_title(ax, f'ESMA Exception Report — Last 250 Days — Zone: {zone_250}', fontsize=12)

ax.set_ylabel('Daily P&L / VaR (%)', fontsize=9)
ax.legend(fontsize=10)
plt.tight_layout()
save_fig(fig, FUND_ID, "05. VAR backtest HF - Last 250d")

plt.show()

## 8. Absolute and Relative VaR Monitoring Report

In [ ]:
# MRS-44: VaR monitoring report
from src.risk.risk_utils import var_scale

# --- Absolute VaR ---
pnl_1y        = returns[-250:]
abs_var_1d    = var_historical(pnl_1y, confidence=0.99)
abs_var_20d   = var_scale(abs_var_1d, horizon=20) * 100  # % NAV
abs_limit     = 20.0
abs_util      = abs_var_20d / abs_limit * 100

# --- Relative VaR ---
rel_var_current = rolling_ratio.iloc[-1] if len(rolling_ratio) > 0 else None
rel_limit       = 2.0
rel_util        = rel_var_current / rel_limit * 100

# --- Summary table ---
summary = pd.DataFrame([
    {'Metric': 'Absolute VaR (20d 99%)',    'Value': f'{abs_var_20d:.2f}%',       'Limit': '20.00%',  'Utilisation': f'{abs_util:.1f}%',  'Status': 'OK' if abs_var_20d < abs_limit else 'BREACH'},
    {'Metric': 'Relative VaR (ratio)',       'Value': f'{rel_var_current:.2f}x',   'Limit': '2.00x',   'Utilisation': f'{rel_util:.1f}%',  'Status': 'OK' if rel_var_current < rel_limit else 'BREACH'},
    {'Metric': 'SRRI',                       'Value': str(srri),                   'Limit': '—',       'Utilisation': '—',                 'Status': '—'},
    {'Metric': 'ESMA Zone (250d)',            'Value': zone_250,                    'Limit': 'Green',   'Utilisation': f'{n_250} breaches', 'Status': 'OK' if zone_250 == 'Green' else 'REVIEW'},
])
summary.set_index('Metric', inplace=True)
summary


In [ ]:
var_lim = 100
var_warning = 80

metrics  = ['Absolute VaR', 'Relative VaR']
utils    = [abs_util, rel_util]
colors   = [ACCENT2 if u > 80 else ACCENT3 if u > 60 else ACCENT for u in utils]

fig, ax = plt.subplots(figsize=(8, 2))
bars = ax.barh(metrics, utils, color=colors, height=0.4, alpha=0.85)

ax.axvline(var_lim, color=ACCENT2, lw=1.5, linestyle='--')
ax.text(var_lim - 1, 0.5, 'Hard limit', color=ACCENT2, fontsize=8,
        va='center', ha='right', rotation=90,
        transform=ax.get_xaxis_transform())

ax.axvline(var_warning, color=C['amber'], lw=1, linestyle='--')
ax.text(var_warning - 1, 0.5, 'Warning', color=C['amber'], fontsize=8,
        va='center', ha='right', rotation=90,
        transform=ax.get_xaxis_transform())

ax.set_xlabel('Limit utilisation (%)', fontsize=9)
section_title(ax, f'VaR Limit Utilisation — {VALUATION_DATE}')
ax.grid(False)
for bar, val in zip(bars, utils):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
save_fig(fig, FUND_ID, "06. VAR Limit Utilisation  - UCITS")
plt.show()



## 9. SRRI Monitoring and KIID Update Trigger

CESR/10-673 requires ManCos to monitor SRRI continuously and update the KIID
if the SRRI changes category for 4 consecutive months. The rolling SRRI is
computed monthly using a trailing 260-week (5-year) window.

In [ ]:
# MRS-45: SRRI monitoring
nav_history_full = query_nav_history(ENGINE, FUND_ID)
nav_history_full['date'] = pd.to_datetime(nav_history_full['date'])
nav_history_full = nav_history_full.set_index('date')

# resample to weekly, compute rolling 260-week SRRI monthly
weekly_nav_full = nav_history_full['nav_eur'].resample('W').last()
weekly_ret_full = weekly_nav_full.pct_change().dropna()

# compute SRRI at each month-end using trailing 260 weeks
monthly_ends = weekly_ret_full.resample('ME').last().index
rolling_srri = []

for dt in monthly_ends:
    window_ret = weekly_ret_full[:dt].iloc[-260:]
    if len(window_ret) < 52:
        continue
    sigma_w   = window_ret.std()
    sigma_a   = sigma_w * np.sqrt(52)
    rolling_srri.append({'date': dt, 'srri': compute_srri(sigma_a), 'sigma_ann': sigma_a})

srri_df = pd.DataFrame(rolling_srri).set_index('date')
print(f"Rolling SRRI observations : {len(srri_df)}")
srri_df.tail(6)

In [ ]:
# flag 4 consecutive months where SRRI stays at a new level
srri_df['srri_prev']     = srri_df['srri'].shift(1)
srri_df['changed']       = srri_df['srri'] != srri_df['srri_prev']

# count consecutive months at current SRRI vs initial SRRI
initial_srri             = srri_df['srri'].iloc[0]
srri_df['consec_new']    = 0

consec = 0
ref    = srri_df['srri'].iloc[0]
for idx, row in srri_df.iterrows():
    if row['srri'] != ref:
        consec += 1
    else:
        consec = 0
        ref    = row['srri']
    srri_df.at[idx, 'consec_new'] = consec

srri_df['kiid_update'] = srri_df['consec_new'] >= 4

n_triggers = srri_df['kiid_update'].sum()
print(f"KIID update triggers in history : {n_triggers}")
print(f"Current SRRI                    : {srri_df['srri'].iloc[-1]}")
print(f"KIID update required now        : {srri_df['kiid_update'].iloc[-1]}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 4), sharex=True)
sup_title(fig, 'SRRI Monitoring', fontsize=15)

# top: rolling annualised volatility
ax1.plot(srri_df.index, srri_df['sigma_ann'] * 100,
      color=ACCENT, lw=1.2, label='Annualised vol (%)')
ax1.set_ylabel('Volatility (%)', fontsize=9)
ax1.grid(True, axis='y', alpha=0.15, linestyle='--')
ax1.tick_params(labelsize=9, length=0)
ax1.legend(fontsize=8)

# bottom: rolling SRRI with KIID triggers
ax2.step(srri_df.index, srri_df['srri'],
             color=ACCENT, lw=1.5, label='SRRI', where='post')
ax2.set_yticks([1, 2, 3, 4, 5, 6, 7])
ax2.set_ylim(0.5, 7.5)
ax2.set_ylabel('SRRI category', fontsize=9)
ax2.grid(True, axis='y', alpha=0.15, linestyle='--')
ax2.tick_params(labelsize=9, length=0)

triggers = srri_df[srri_df['kiid_update']]
if len(triggers) > 0:
    ax2.scatter(triggers.index, triggers['srri'],
                    color=ACCENT2, s=5, zorder=5,
                    label=f'KIID update trigger ({len(triggers)})')
ax2.legend(fontsize=8)

plt.tight_layout()
save_fig(fig, FUND_ID, "07. SRRI Monitoring")
plt.show()



## 10. Monthly Risk Report

Summary risk report as of the most recent valuation date.
Produced monthly by the Risk Management function and reviewed by the Board.

In [ ]:
# MRS-46: monthly risk report
from datetime import datetime

report_date = pd.Timestamp(VALUATION_DATE).strftime('%B %d, %Y')
nav_eur     = risk_df['market_value_eur'].sum()

print('=' * 60)
print(f'  UCITS BALANCED FUND — MONTHLY RISK REPORT')
print(f'  Valuation date : {report_date}')
print(f'  NAV            : EUR {nav_eur:,.0f}')
print('=' * 60)

print('\n  1. VAR SUMMARY')
print('  ' + '-' * 40)
print(f'  Absolute VaR (20d 99%)  : {abs_var_20d:.2f}%   limit 20.00%   util {abs_util:.1f}%')
print(f'  Relative VaR (ratio)    : {rel_var_current:.2f}x     limit 2.00x     util {rel_util:.1f}%')
print(f'  VaR model               : Historical simulation, 250-day window')

print('\n  2. SRRI')
print('  ' + '-' * 40)
print(f'  Current SRRI            : {srri_df["srri"].iloc[-1]}')
print(f'  Annualised volatility   : {srri_df["sigma_ann"].iloc[-1]*100:.2f}%')
print(f'  KIID update required    : {"YES — action required" if srri_df["kiid_update"].iloc[-1] else "No"}')

print('\n  3. BACKTEST (ESMA)')
print('  ' + '-' * 40)
print(f'  Observation window      : 250 trading days')
print(f'  Breaches                : {n_250}')
print(f'  Expected breaches       : 2.5 (1% x 250)')
print(f'  ESMA zone               : {zone_250}')
print(f'  Kupiec p-value          : {report["kupiec_p"].iloc[0]:.4f}')
print(f'  Christoffersen p-value  : {report["christoffersen_p"].iloc[0]:.4f}')

print('\n  4. STRESS TESTING')
print('  ' + '-' * 40)
print(f'  Equity crash -30%       : {eq_pct:.2f}% NAV')
print(f'  Rate shock +200bps      : {rt_pct:.2f}% NAV')
print(f'  Credit widening +150bps : {cr_pct:.2f}% NAV')
print(f'  FX stress -15%          : {fx_pct:.2f}% NAV')
print(f'  Combined scenario       : {cb_pct:.2f}% NAV')
print(f'  2008 financial crisis   : {pnls_pct[0]:.2f}% NAV')
print(f'  2020 COVID crash        : {pnls_pct[1]:.2f}% NAV')
print(f'  2022 rate shock         : {pnls_pct[2]:.2f}% NAV')

print('\n  5. COMPLIANCE')
print('  ' + '-' * 40)
abs_status = 'COMPLIANT' if abs_var_20d < abs_limit else 'BREACH'
rel_status = 'COMPLIANT' if rel_var_current < rel_limit else 'BREACH'
print(f'  Absolute VaR limit      : {abs_status}')
print(f'  Relative VaR limit      : {rel_status}')
print(f'  ESMA backtest zone      : {zone_250}')
print(f'  UCITS eligible          : Yes')

print('\n' + '=' * 60)
print('  Prepared by  : Risk Management')
print(f'  Report date  : {pd.Timestamp(VALUATION_DATE).strftime("%B %d, %Y")}')
print('=' * 60)

In [ ]:
# MRS-31: Redemption stress — UCITS Balanced
# Compute liquidity buckets (risk_df already defined in Section 6)
_ucits_liq = days_to_liquidate(risk_df, pct_adv=0.25)
_ucits_liq = liquidity_buckets(_ucits_liq)
NOTICE = 5   # contractual notice period (days)
_SCENARIOS = [
    (0.10, 'Normal    (10% NAV)'),
    (0.25, 'Large     (25% NAV)'),
    (0.50, 'Stress    (50% NAV)'),
]

print(f'Fund: UCITS Balanced  |  NAV: EUR {NAV:,.0f}  |  Notice: {NOTICE} days')
print()
print(f"{'':22} {'Redemption (M)':>14} {'Liquid (M)':>12} {'Gap (M)':>12} {'Coverage':>10} Action")
print('\u2500' * 95)

_redstress = {}
for _pct, _label in _SCENARIOS:
    _r = redemption_stress(_ucits_liq, NAV, redemption_pct=_pct, notice_days=NOTICE)
    _redstress[_pct] = _r
    _gap = f"+{_r['liquidity_gap_eur']/1e6:.1f}M" if _r['liquidity_gap_eur'] >= 0 else f"{_r['liquidity_gap_eur']/1e6:.1f}M"
    print(f"{_label:<22} {_r['redemption_amount_eur']/1e6:>13.1f}M {_r['liquid_assets_eur']/1e6:>11.1f}M "
          f"{_gap:>12} {_r['coverage_ratio']:>9.2f}x  {_r['recommendation']}")
print('\u2500' * 95)
print('Largest-investor stress: see Section 6.3')

### 6.3 Investor Concentration — MRS-32

**ESMA thresholds** (ESMA/2020/1498, Annex VI):
- Single investor **> 20% of NAV** → concentration risk flag
- Top 3 investors **> 50% of NAV** → high-concentration flag

A concentrated investor base amplifies redemption risk: one large exit
can force asset sales that affect all remaining investors. The largest-investor
scenario below connects MRS-31 and MRS-32 — it uses the investor register to
derive the fourth redemption stress scenario.

In [ ]:
# MRS-32: Investor concentration — UCITS Balanced
# UCITS: 500+ retail investors — representative sample, top 10 + aggregate
_investors = pd.DataFrame([
    {'investor_id': 'UC001', 'investor_name': 'Pension Plan A',           'investor_type': 'Pension Plan', 'aum_eur': NAV * 0.0200},
    {'investor_id': 'UC002', 'investor_name': 'Distribution Platform B',  'investor_type': 'Platform',     'aum_eur': NAV * 0.0170},
    {'investor_id': 'UC003', 'investor_name': 'Insurance Wrapper C',      'investor_type': 'Insurance',    'aum_eur': NAV * 0.0130},
    {'investor_id': 'UC004', 'investor_name': 'Retail Investor D',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0080},
    {'investor_id': 'UC005', 'investor_name': 'Retail Investor E',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0070},
    {'investor_id': 'UC006', 'investor_name': 'Retail Investor F',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0065},
    {'investor_id': 'UC007', 'investor_name': 'Retail Investor G',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0060},
    {'investor_id': 'UC008', 'investor_name': 'Retail Investor H',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0055},
    {'investor_id': 'UC009', 'investor_name': 'Retail Investor I',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0050},
    {'investor_id': 'UC010', 'investor_name': 'Retail Investor J',        'investor_type': 'Retail',       'aum_eur': NAV * 0.0045},
    {'investor_id': 'UC_REM','investor_name': 'Remaining (490+ retail)', 'investor_type': 'Retail',       'aum_eur': NAV * 0.9075},
])

_conc = investor_concentration(_investors, NAV, threshold=0.20)
_top  = _conc['by_investor']
_type = _investors.set_index('investor_id')['investor_type']

print(f'Investor Concentration — UCITS Balanced  |  NAV: EUR {NAV:,.0f}')
print('ESMA threshold: 20% single / 50% top-3\n')
print(f"{'':4} {'Investor':<30} {'Type':<18} {'AUM (EUR)':>14} {'% NAV':>8}")
print('\u2500' * 80)
for _rank, (_, _row) in enumerate(_top.iterrows(), 1):
    _t = _type.get(_row['investor_id'], '')
    print(f"{_rank:<4} {_row['investor_name']:<30} {_t:<18} {_row['aum_eur']:>14,.0f} {_row['pct_nav']*100:>7.1f}%")
print('\u2500' * 80)

_flag_s = '\u26a0 ESMA flag'       if _conc['concentration_flag'] else '\u2713 OK'
_flag_3 = '\u26a0 High conc.'      if _conc['high_concentration'] else '\u2713 OK'
print(f"\nLargest investor : {_conc['largest_investor_pct']*100:.1f}% NAV  {_flag_s}")
print(f"Top 3 investors  : {_conc['top3_pct']*100:.1f}% NAV  {_flag_3}")

# Largest-investor redemption stress (4th scenario)
_r4   = redemption_stress(_ucits_liq, NAV, redemption_pct=_conc['largest_investor_pct'], notice_days=5)
_gap4 = f"+{_r4['liquidity_gap_eur']/1e6:.1f}M" if _r4['liquidity_gap_eur'] >= 0 else f"{_r4['liquidity_gap_eur']/1e6:.1f}M"
print(f"\nLargest-investor stress ({_conc['largest_investor_pct']*100:.1f}% NAV, 5-day notice):")
print(f"  Redemption : EUR {_r4['redemption_amount_eur']:,.0f}")
print(f"  Liquid     : EUR {_r4['liquid_assets_eur']:,.0f}")
print(f"  Gap        : {_gap4}  |  Coverage: {_r4['coverage_ratio']:.2f}x")
print(f"  Action     : {_r4['recommendation']}")

print('\nMonitoring recommendation:')
if _conc['high_concentration']:
    print('  \u2014 Enhanced monitoring: top-3 investors represent significant co-ordinated exit risk')
    print('  \u2014 Maintain liquidity buffer >= largest investor AUM')
if _conc['concentration_flag']:
    print(f'  \u2014 Gate-trigger review: largest investor at {_conc["largest_investor_pct"]*100:.1f}% NAV')
if not _conc['concentration_flag'] and not _conc['high_concentration']:
    print('  \u2014 No immediate action. Continue quarterly investor concentration monitoring.')

### 6.4 Counterparty Risk

UCITS Article 52 caps OTC derivative counterparty exposure at 10% of NAV for credit 
institutions and 5% for all others. The relevant exposure is the net uncollateralised 
position after netting initial and variation margin held at the counterparty.

> Collateral coverage is simulated. A production system would derive these figures from 
the daily collateral reconciliation.

In [ ]:
# MRS-XX: Counterparty risk — UCITS
# Simulated OTC derivatives counterparty register
_cp_ucits = rk.load_counterparty(FUND_ID)
_cp_ucits['exposure_eur']     = _cp_ucits['exposure_pct'] * NAV
_cp_ucits['collateral_eur']   = _cp_ucits['exposure_eur'] * _cp_ucits['collateral_cover']
_cp_ucits['net_exposure_eur'] = _cp_ucits['exposure_eur'] * (1 - _cp_ucits['collateral_cover'])
_cp_ucits['net_pct_nav']      = _cp_ucits['net_exposure_eur'] / NAV
_cp_ucits['limit_pct']        = _cp_ucits['type'].map(
                                    {'credit_institution': 0.10, 'other': 0.05})
_cp_ucits['breach']           = _cp_ucits['net_pct_nav'] > _cp_ucits['limit_pct']

_worst_cp    = _cp_ucits.loc[_cp_ucits['net_exposure_eur'].idxmax()]
_cp_loss_eur = _worst_cp['net_exposure_eur']
_cp_loss_pct = _worst_cp['net_pct_nav']

phtml.display_counterparty_risk_ucits(NAV, _cp_ucits, _worst_cp, _cp_loss_eur, _cp_loss_pct)

## 12. Pre-Trade Compliance Check

For UCITS, pre-trade compliance is a statutory obligation. \
UCITSD Articles 50, 52 and 83 impose hard limits that a ManCo cannot waive — the portfolio \
manager must receive a compliance sign-off before any trade that could affect eligibility, \
concentration, or VaR.

`pre_trade_check` runs the following checks for UCITS:
- **Absolute VaR** post-trade < 20% NAV (SRRI framework, 20-day, 99%)
- **Relative VaR** < 2× reference portfolio (60/40 benchmark)
- **5/10/40 rule** (Article 52): single issuer < 10% NAV; issuers > 5% must aggregate to < 40%
- **Eligible assets** (Article 50): no direct real estate, loans, CLOs, private equity
- **Counterparty exposure** (OTC derivatives): < 10% for credit institutions, < 5% otherwise
- **Borrowing limit** < 10% NAV (Article 83 — temporary cash borrowing only)

> **ETF look-through limitation.** This portfolio is dominated by broad index ETFs \
(SPY 48%, Euro Stoxx 28%). In a fully compliant implementation (ESMA Q&A UCITS / \
CESR/10-049), these ETFs would be evaluated on a look-through basis — their underlying \
constituents are checked, not the ETF wrapper itself. The simplified `pre_trade_check` \
applies the 5/10/40 rule at ETF level, which means the existing SPY and Euro Stoxx positions \
appear as pre-existing issuer concentration flags. The relevant compliance question for each \
scenario below is whether the *proposed trade* introduces a new or worsened breach relative \
to the current portfolio state.

In [ ]:
from src.risk.risk_utils import pre_trade_check

### 12.1  Scenario 1 — Small IG bond buy

Buy BMW AG bonds, new issuer not currently in the portfolio. \
EUR 5M notional on a EUR 515M NAV = 0.97% of NAV. \
The trade is small and compliant. The check will surface the pre-existing SPY and \
Euro Stoxx concentrations (ETF look-through limitation), but the proposed trade does not \
introduce any new or worsened breach — which is the material compliance question.\

In [ ]:
# Pre-trade check context (UCITS version — no derivatives)

PTC_CTX = dict(
    engine            = ENGINE,
    fund_id           = FUND_ID,
    date              = VALUATION_DATE,
    counterparties_df = _cp_ucits,
    bbg               = BBG,
)

In [ ]:
# BMW AG IG bond — new issuer, small position, eligible instrument
trade_bond = {
    'isin'           : 'XS2600000001',
    'direction'      : 'buy',
    'quantity'       : 5_000_000,         # EUR 5 M at par
    'price_eur'      : 1.00,
    'asset_class'    : 'Bond',
    'sub_asset_class': 'IG Corporate',
    'dur_adj_mid'    : 3.5,
    'currency'       : 'EUR',
    'adv_eur'        : 10_000_000,
    'issuer'         : 'BMW AG',}



result_conc = pre_trade_check(trade_bond, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_conc)

### 12.2  Scenario 2 — Single-stock concentration breach (5/10/40 rule, Article 52)

Buy Apple Inc (not in the UCITS portfolio), 350,000 shares at EUR 172/share \
= EUR 60.2M notional. On NAV of EUR 514.8M, this is **11.7%** of NAV — \
a clear breach of the 10% single-issuer hard limit under UCITSD Article 52. \
A UCITS ManCo must block this order.

The 10% limit applies per *issuer* of transferable securities, not per instrument. \
If the same issuer were held in both equity and bond form, both exposures would be \
aggregated for the 5/10/40 test. A UCITS portfolio can hold up to six issuers at 5–10% \
(the "40 rule" caps the aggregate of all positions > 5%), but no single issuer may exceed 10%.

In [ ]:
# Apple Inc — new single-stock position sized to breach 10% issuer limit (11.7% NAV)
trade_conc = {
    'isin'           : 'US0231351067',
    'direction'      : 'buy',
    'quantity'       : 350_000,           # 350 k × EUR 172 = EUR 60.2 M
    'price_eur'      : 172.00,
    'asset_class'    : 'Equity',
    'sub_asset_class': 'Large Cap',
    'beta'           : 1.25,
    'currency'       : 'USD',
    'adv_eur'        : 200_000_000,
    'issuer'         : 'Apple Inc',
}

result_conc = pre_trade_check(trade_conc, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_conc)

### 12.3  Scenario 3 — Ineligible asset (UCITSD Article 50)

The portfolio manager proposes to invest EUR 10M in a direct real estate holding. \
Direct property is not a transferable security, money market instrument, unit in a UCITS/UCI, \
bank deposit, or eligible derivative — it is therefore ineligible under UCITSD Article 50. \
A UCITS ManCo must refuse this trade. There is no discretion here; this is a statutory \
prohibition, not an internal risk limit.

In [ ]:
# Direct property — ineligible under UCITSD Art. 50
trade_inelig = {
    'isin'           : 'LU-PROP-DIRECT-001',
    'direction'      : 'buy',
    'quantity'       : 1,
    'price_eur'      : 10_000_000,
    'asset_class'    : 'Real Estate',
    'sub_asset_class': 'Direct Property',
    'currency'       : 'EUR',
    'adv_eur'        : 0,
    'issuer'         : 'Luxembourg Office Holdings SA',
}

result_inelig = pre_trade_check(trade_inelig, ENGINE, FUND_ID, VALUATION_DATE)
phtml.display_ptc(result_inelig)

## ESG Risk Indicators

ESG monitoring is required under CSSF Regulation 22-05 and SFDR (EU 2019/2088).
Portfolio-level ESG scores are computed as NAV-weighted averages. 

**SFDR classification**: Article 6 (no sustainability claim). ESG factors are
monitored but do not drive investment decisions. Article 8 would require documented
ESG screening; Article 9 would require sustainable investment as the primary objective.

Metrics monitored:
- Weighted average ESG score (composite, E, S, G)
- % of NAV in instruments with ESG score below 30*
- % of NAV with active controversy flag
- Weighted average carbon intensity (tCO2/EURm revenue)

> **Note**: ESG scores for liquid instruments are fetched from Bloomberg at
> enrichment time. Instruments without a Bloomberg ticker use fund administrator
> ESG data embedded in the position file.


> % of NAV in instruments with ESG score below internal threshold. 
> Threshold is defined in the Risk Management Policy. 
> ESG scores are not comparable across providers as each
> uses a different methodology and scale.
> (here using 30/100,Bloomberg scale 0-100 higher is better)

> **Scale note**: all ESG scores in this framework use a 0-100 scale where
> 100 is best, consistent with Bloomberg ESG methodology. Illiquid instrument
> scores are provided by the fund administrator on the same scale. In production,
> the ManCo should ensure all third-party ESG data is normalised to a consistent
> scale before aggregating portfolio-level metrics.

> **ESG look-through for derivatives**: derivatives have indirect ESG exposure
> through their underlying. The delta-adjusted notional is used as the ESG
> exposure weight rather than market value, which would understate the exposure.
>
> For options:
> $$ESG\_exposure_i = |\delta_i| \times Q_i \times \text{contract size} \times P_{underlying} \times FX$$

> For futures: full notional is used since delta = 1.
> For FX forwards: no ESG exposure assigned (no issuer).

In [ ]:
# esg_df  = build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
# summary = esg_portfolio_summary(esg_df, NAV)

import src.risk.esg_utils as esg_u 
esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df)

In [ ]:
esg_u.display_esg_summary(esg_df)

In [ ]:
esg_u.plot_esg_profile(esg_df, FUND_ID, plot_title='ESG profile - UCITS Balanced')

## 11. P&L Attribution by Risk Factor

Under AIFMD Article 15 and CSSF expectations for risk governance, the risk
function should be able to explain return drivers by factor. For UCITS this
is an internal governance output, not a direct regulatory deliverable.

The **value bridge methodology** decomposes daily P&L into:
* market beta (systematic equity exposure)
* rates (duration × parallel yield shift)
* FX (non-EUR position exposure)
* residual (idiosyncratic / unexplained)

Benchmark: SX5E (Euro Stoxx 50) — appropriate for this EUR-denominated
balanced fund. The residual reflects alpha from security selection and
factors not captured by the single-index model.

> Sensitivity-based attribution uses actual daily positions and actual
> market moves, consistent with how VaR is computed.

In [ ]:
import sqlalchemy as sa
from src.risk.risk_utils import compute_pnl_attribution
from src.data.database import query_nav_history

# Actual daily P&L
nav_history = query_nav_history(ENGINE, FUND_ID)
nav_history['date'] = pd.to_datetime(nav_history['date'])
nav_history = nav_history.set_index('date').sort_index()
pnl_actual  = nav_history['pnl_eur'].dropna()

START_DATE         = pnl_actual.index.min().strftime('%Y-%m-%d')
VALUATION_DATE_STR = VALUATION_DATE

# Benchmark — SPY for USD-heavy long/short portfolio
spy_bm = BBG.bdh('SPY US Equity', 'PX_LAST', START_DATE, VALUATION_DATE_STR)
spy_bm['r_market'] = spy_bm['PX_LAST'].pct_change()

# Rate shift — simulated parallel USD curve move
np.random.seed(42)
rate_series = pd.Series(
    np.random.normal(0, 0.0005, len(spy_bm)),
    index=spy_bm.index, name='dy'
)

# FX
usd = BBG.bdh('EURUSD Curncy', 'PX_LAST', START_DATE, VALUATION_DATE_STR)
usd['r_fx_USD'] = usd['PX_LAST'].pct_change()

# Market moves
market_moves = pd.DataFrame(index=spy_bm.index)
market_moves['r_market'] = spy_bm['r_market']
market_moves['dy']       = rate_series
market_moves['r_fx_USD'] = usd['r_fx_USD']
market_moves = market_moves.dropna()

# Daily positions history
with ENGINE.connect() as conn:
    positions_history = pd.read_sql(sa.text("""
        SELECT p.date, p.isin, p.asset_class, p.currency,
               p.market_value_eur, pe.beta, pe.dur_adj_mid
        FROM positions p
        LEFT JOIN positions_enriched pe
            ON p.isin = pe.isin AND p.fund_id = pe.fund_id
        WHERE p.fund_id = :fund_id
        ORDER BY p.date
    """), conn, params={'fund_id': FUND_ID})

positions_history['date'] = pd.to_datetime(positions_history['date'])

common_dates     = market_moves.index.intersection(pnl_actual.index)
market_moves_aln = market_moves.loc[common_dates]
returns_aligned      = pnl_actual.loc[common_dates]
pos_history_aln  = positions_history[positions_history['date'].isin(common_dates)]

attr = compute_pnl_attribution(pos_history_aln, market_moves_aln, returns_aligned)

# Cumulative attribution chart
fig, ax = plt.subplots(figsize=(8, 4))
cum = attr[['pnl_equity', 'pnl_rates', 'pnl_fx', 'pnl_residual']].cumsum() / 1e6
ax.plot(cum.index, cum['pnl_equity'],   color=ACCENT,    linewidth=1.5, label='Equity')
ax.plot(cum.index, cum['pnl_rates'],    color=ACCENT2,   linewidth=1.5, label='Rates')
ax.plot(cum.index, cum['pnl_fx'],       color=ACCENT3,   linewidth=1.5, label='FX')
ax.plot(cum.index, cum['pnl_residual'], color=C['red'], linewidth=1.0,
        linestyle='--', label='Residual')
ax.axhline(0, color='white', linewidth=0.5, linestyle='--')
ax.set_ylabel('Cumulative P&L (EUR MM)')
section_title(ax, f'Cumulative P&L Attribution by Risk Factor — UCITS', fontsize=16)
ax.legend(fontsize=9)
plt.tight_layout()
save_fig(fig, FUND_ID, "09. PnL attribution UCITS")
plt.show()


# Model quality
RESIDUAL_THRESHOLD_PCT = 0.20
resid_vol = attr['pnl_residual'].std()
flagged = attr[
    (attr['pct_explained'].notna()) &
    (
        (1 - attr['pct_explained'] > RESIDUAL_THRESHOLD_PCT) |
        (attr['pnl_residual'].abs() > 2.0 * resid_vol)
    )
].copy()

print(f"{'Attribution period':<35} {attr.index.min().date()} to {attr.index.max().date()}")
print(f"{'Days attributed':<35} {len(attr)}")
print(f"{'Correlation (actual vs expl.)':<35} {attr['pnl_actual'].corr(attr['pnl_explained']):.3f}")
print(f"{'Median % explained':<35} {attr['pct_explained'].median():.1%}")
print(f"{'Days >= 80% explained':<35} {(attr['pct_explained'] >= 0.80).sum()} ({(attr['pct_explained'] >= 0.80).mean():.1%})")
print(f"{'Residual vol (EUR)':<35} {attr['pnl_residual'].std():,.0f}")
print(f"{'Residual / total vol':<35} {attr['pnl_residual'].std() / attr['pnl_actual'].std():.1%}")
print(f"{'Flagged days':<35} {len(flagged)} ({len(flagged)/len(attr):.1%})")
print()
print("Note: residual = idiosyncratic return not explained by market beta.")
print("For a long/short equity fund the residual represents alpha —")
print("stock-specific return from security selection independent of")
print("market direction. Persistent positive residual = successful stock picking.")




**Methodology and limitations**

Sensitivity-based attribution using actual daily positions and market moves.
Regression-based attribution is not used — it gives average historical loadings
and cannot reflect current position composition.

Benchmark: SX5E (Euro Stoxx 50) — appropriate for this EUR-denominated balanced fund.

**Limitations:**
* Single-factor equity model — no sector, style, or stock-level decomposition
* Rate attribution uses a simulated parallel shift — no key-rate DV01
* FX covers EUR/USD only
* Position composition is static in this simulation

**Regulatory context:** this section is an internal governance output consumed
by the Board and risk committee. It supports the AIFMD Article 15 evidence
pack and CSSF expectations for risk management reporting. It is not a direct
Annex IV or Annex VI deliverable.